# Module 4: Building a Comparison Group

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

Every method from here on needs a comparison group. This module builds one,
writes down the rule that produced it, and checks how much the answer depends
on the rule.

The habit being taught is narrow and important: **decide the rule before
looking at the outcome, then report what other defensible rules would have
given.**

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

## 2. What a rule has to do

A rule for admitting agencies to the comparison group has three jobs.

| Job | Why |
|---|---|
| **Exclude anyone who got the program** | including under another name |
| **Be statable in one sentence, before results** | so nobody can adjust it afterwards |
| **Admit enough agencies to be steady** | one agency's accidents become the estimate |

Notice what is **not** on the list: resembling the treated agencies. Similarity
helps only when it makes the trends more alike, which is a different and much
narrower claim. [Module 3](Module_03_The_Counterfactual_You_Have_To_Construct.ipynb)
showed matching on size making things worse.

In [ ]:
prof = profile.set_index("agency_id")
print("  the seven agencies that did not take the training\n")
display(prof.loc[COMPARISON,
                 ["agency_name", "agency_type", "sworn_officers",
                  "population_served", "cfs_data_submitted"]]
        .rename(columns={"cfs_data_submitted": "call data"}))

## 3. Four defensible rules

Each of these could be written down in advance and defended to a reviewer.

In [ ]:
rules = {
    "everyone not trained": COMPARISON,
    "municipal police only":
        [a for a in COMPARISON if prof.loc[a, "agency_type"] == "Municipal Police"],
    "at least 30 sworn officers":
        [a for a in COMPARISON if prof.loc[a, "sworn_officers"] >= 30],
    "complete call data":
        [a for a in COMPARISON if prof.loc[a, "cfs_data_submitted"] == "Yes"],
}
for name, members in rules.items():
    print(f"  {name:28s} n = {len(members)}   "
          f"{', '.join(NAME[a].split()[0] for a in members)}")

## 4. What each rule gives

In [ ]:
keep = [a for a in TRAINED if a != "A007"]
tb, ta = cell_rate(keep, "before"), cell_rate(keep, "after")

rows = []
for name, members in rules.items():
    cb, ca = cell_rate(members, "before"), cell_rate(members, "after")
    rows.append({"rule": name, "agencies": len(members),
                 "comparison changed": f"{100 * (ca / cb - 1):+.1f}%",
                 "estimate": f"{100 * ((ta / tb) / (ca / cb) - 1):+.1f}%"})
rows.append({"rule": "THE TRUTH", "agencies": "", "comparison changed": "",
             "estimate": f"{TRUTH:+.1f}%"})
pd.DataFrame(rows).set_index("rule")

Every rule lands between 12.0 and 12.8 percent, a spread of **0.8 points**.

That is the result you hope for and must never assume. It says the answer does
not hinge on a judgment call, which is exactly the thing a sceptical reader
wants to know and cannot find out from a single number.

**Report this table, not just the row you used.** It costs four lines of code
and it removes the most common objection to any comparison group.

## 5. When the rules disagree

A spread of 0.8 points is reassuring. A spread of 10 would mean the estimate is
a judgment call, and there is a right way to handle that too.

In [ ]:
bad_rules = {
    "the three smallest agencies":
        sorted(COMPARISON, key=lambda a: prof.loc[a, "sworn_officers"])[:3],
    "the three largest agencies":
        sorted(COMPARISON, key=lambda a: -prof.loc[a, "sworn_officers"])[:3],
    "one agency, chosen for being nearby": ["A009"],
}
rows = []
for name, members in bad_rules.items():
    cb, ca = cell_rate(members, "before"), cell_rate(members, "after")
    rows.append({"rule": name, "agencies": len(members),
                 "estimate": f"{100 * ((ta / tb) / (ca / cb) - 1):+.1f}%"})
print(f"  the truth is {TRUTH:+.1f} percent\n")
pd.DataFrame(rows).set_index("rule")

Now the spread is enormous, and one rule produces the wrong sign.

The pattern is not simply how many agencies each rule admits. The three
largest agencies, only three of them, give 12.5 percent, as good as any rule
above. The three smallest, also three, give 20.5.

**What matters is how many incidents the comparison group contains, not how
many agencies.** Three small departments together record fewer incidents in
four years than one large one records in a few months, so their pooled rate
still swings on accidents. Count incidents, not logos.

## 6. The rule to write down

For this dataset, and for most WADEPS work:

> *All agencies that did not adopt the program, excluding any that adopted a
> comparable program under another name, and excluding any whose pre program
> trend differs significantly from the treated group. Exclusions listed
> individually with reasons.*

That sentence is checkable, it was written before any estimate, and every
departure from it is visible.

## Exercise

Every rule above kept Ashfell, which has 902 officers and dominates the pooled
comparison group. Find out how much of the estimate is Ashfell.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    rows = []
    for drop in [None] + COMPARISON:
        members = [a for a in COMPARISON if a != drop]
        cb, ca = cell_rate(members, "before"), cell_rate(members, "after")
        rows.append({"agency dropped": "none" if drop is None else NAME[drop],
                     "officers": "" if drop is None else f"{prof.loc[drop, 'sworn_officers']:.0f}",
                     "estimate": round(100 * ((ta / tb) / (ca / cb) - 1), 1)})
    out = pd.DataFrame(rows).set_index("agency dropped")
    print(f"  the truth is {TRUTH:+.1f} percent\n")
    display(out)
    print(f"  largest single agency influence: "
          f"{(out['estimate'] - out.loc['none', 'estimate']).abs().max():.1f} points")
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Dropping Ashfell moves the estimate from 12.5 percent to **14.7 percent**, a
swing of 2.2 points. Every other agency moves it by less than 0.7.

The reason is in one number: **Ashfell accounts for 76.8 percent of all the
use of force incidents in the comparison group.** The pooled comparison rate
is very nearly Ashfell's own rate wearing six other names. If Ashfell had
behaved unusually over these years, the estimate would have followed it, and
nothing in the summary table of [section 4](#4-what-each-rule-gives) would
have shown that.

This is a leave one out check and it takes five lines. Run it whenever one
member of a pooled group is much larger than the rest, and report the largest
movement it produces.

A weighted alternative exists, giving each agency equal weight rather than
weight proportional to its size, and it answers a slightly different question:
the average effect across agencies rather than across incidents. Neither is
wrong. They should not be confused, and the report should say which was used.

</details>

---

**Next:** [Module 5: Difference in Differences, by Hand](Module_05_Difference_In_Differences_By_Hand.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*